# Data processing - shots

In [7]:
import pandas as pd

DATA_PATH = "../../data/"

In [8]:
def get_season(date):
    year = date.year
    return year if date.month >= 9 else year - 1

## Load shot detail data

In [9]:
import glob

shot_files = sorted(glob.glob(DATA_PATH + "/nba_play_by_play_shot_data/shotdetail*.csv"))

len(shot_files)

58

## Process data

In [10]:
# GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,
# EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM

# hoop is at (LOC_X, LOC_Y) = (0, 0)

cols_to_keep = [
    "GAME_ID", "GAME_DATE", "PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_NAME",
    "PERIOD", "MINUTES_REMAINING", "SECONDS_REMAINING",
    "SHOT_TYPE", "SHOT_DISTANCE", "LOC_X", "LOC_Y",
    "SHOT_ATTEMPTED_FLAG", "SHOT_MADE_FLAG"
]

frames = []

for file_path in shot_files:
    # only regular season data; "_po_" files are playoffs
    if "_po_" in file_path.lower():
        continue

    df = pd.read_csv(file_path, usecols=lambda c: c in cols_to_keep)

    # 1 = actual shot attempt
    if "SHOT_ATTEMPTED_FLAG" in df.columns:
        df = df[df["SHOT_ATTEMPTED_FLAG"] == 1]

    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y%m%d", errors="coerce")
    df = df[df["GAME_DATE"].notna()]
    df["season"] = df["GAME_DATE"].apply(get_season)

    # cleanup corrdinates
    df["LOC_X"] = pd.to_numeric(df["LOC_X"], errors="coerce")
    df["LOC_Y"] = pd.to_numeric(df["LOC_Y"], errors="coerce")
    df = df.dropna(subset=["LOC_X", "LOC_Y"])

    frames.append(df)

shot_events = pd.concat(frames, ignore_index=True)

shot_events = shot_events.rename(columns={
    "GAME_ID": "gameId",
    "GAME_DATE": "gameDate",
    "PLAYER_ID": "personId",
    "TEAM_ID": "teamId",
    "MINUTES_REMAINING": "minutesRemaining",
    "SECONDS_REMAINING": "secondsRemaining",
    "SHOT_TYPE": "shotType",
    "LOC_X": "locX",
    "LOC_Y": "locY",
    "SHOT_MADE_FLAG": "shotMade",
})

# SHOT_TYPE: 3PT Field Goal or 2PT Field Goal
shot_events["shotValue"] = shot_events["shotType"].str.extract(r"(\d)PT")[0].astype("Int64")

shot_events = shot_events[[
    "season", "gameId", "gameDate",
    "personId", "teamId",
    "minutesRemaining", "secondsRemaining",
    "shotValue", "locX", "locY", "shotMade"
]]

shot_events.head()

,season,gameId,gameDate,personId,teamId,minutesRemaining,secondsRemaining,shotValue,locX,locY,shotMade
0,1996,29600005,1996-11-01,120,1610612737,11,8,2,0,0,1
1,1996,29600005,1996-11-01,120,1610612737,10,32,2,153,71,0
2,1996,29600005,1996-11-01,363,1610612737,9,38,2,-103,26,1
3,1996,29600005,1996-11-01,120,1610612737,8,57,2,0,0,0
4,1996,29600005,1996-11-01,87,1610612737,8,56,2,0,0,0


### Check range for locX, locY

In [11]:
print("locX:", (shot_events["locX"].min(), shot_events["locX"].max()))
print("locY:", (shot_events["locY"].min(), shot_events["locY"].max()))

locX: (np.int64(-250), np.int64(250))
locY: (np.int64(-52), np.int64(884))


## Save data

In [16]:
import numpy as np
from pathlib import Path

output_dir = Path(DATA_PATH) / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

parts = np.array_split(shot_events, 3)

for i, part in enumerate(parts, 1):
    part.to_csv(output_dir / f"shot_events_part{i}.csv", index=False)

/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
